# Limpeza de Dados — Global Superstore 2016

Dataset real: pedidos de uma rede varejista global (2011-2015), carregado diretamente do GitHub.

Fluxo desta aula:
1. Inspeção dos dados
2. Duplicatas
3. Valores faltantes
4. Formatos inconsistentes
5. Conversão de tipos
6. Validação do resultado

Execute as células com `Shift+Enter`, uma de cada vez (igual ao Google Colab).

In [1]:
import pandas as pd
import numpy as np

URL_DADOS = "https://raw.githubusercontent.com/andrewmanueld/dataset_global_superstore_2016/main/CSV/global_superstore_2016_orders.csv"

df = pd.read_csv(URL_DADOS)
df.shape

(51290, 24)

## 1. Inspeção dos dados

Antes de mexer em qualquer coisa, entender o formato bruto: tamanho, colunas, tipos e uma amostra das linhas.

In [2]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Postal Code,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,40098,CA-2014-AB10015140-41954,11/11/2014,11/13/2014,First Class,AB-100151402,Aaron Bergman,Consumer,73120.0,Oklahoma City,...,TEC-PH-5816,Technology,Phones,Samsung Convoy 3,$221.98,2,0.0,$62.15,40.77,High
1,26341,IN-2014-JR162107-41675,2/5/2014,2/7/2014,Second Class,JR-162107,Justin Ritter,Corporate,NaN,Wollongong,...,FUR-CH-5379,Furniture,Chairs,"Novimex Executive Leather Armchair, Black","$3,709.40",9,0.1,$-288.77,923.63,Critical
2,25330,IN-2014-CR127307-41929,10/17/2014,10/18/2014,First Class,CR-127307,Craig Reiter,Consumer,NaN,Brisbane,...,TEC-PH-5356,Technology,Phones,"Nokia Smart Phone, with Caller ID","$5,175.17",9,0.1,$919.97,915.49,Medium
3,13524,ES-2014-KM1637548-41667,1/28/2014,1/30/2014,First Class,KM-1637548,Katherine Murray,Home Office,NaN,Berlin,...,TEC-PH-5267,Technology,Phones,"Motorola Smart Phone, Cordless","$2,892.51",5,0.1,$-96.54,910.16,Medium
4,47221,SG-2014-RH9495111-41948,11/5/2014,11/6/2014,Same Day,RH-9495111,Rick Hansen,Consumer,NaN,Dakar,...,TEC-CO-6011,Technology,Copiers,"Sharp Wireless Fax, High-Speed","$2,832.96",8,0.0,$311.52,903.04,Critical


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  object 
 2   Order Date      51290 non-null  object 
 3   Ship Date       51290 non-null  object 
 4   Ship Mode       51290 non-null  object 
 5   Customer ID     51290 non-null  object 
 6   Customer Name   51290 non-null  object 
 7   Segment         51290 non-null  object 
 8   Postal Code     9994 non-null   float64
 9   City            51290 non-null  object 
 10  State           51290 non-null  object 
 11  Country         51290 non-null  object 
 12  Region          51290 non-null  object 
 13  Market          51290 non-null  object 
 14  Product ID      51290 non-null  object 
 15  Category        51290 non-null  object 
 16  Sub-Category    51290 non-null  object 
 17  Product Name    51290 non-null 

In [ ]:
#resumo de tipos e nulos por coluna, lado a lado
resumo = pd.DataFrame({
    "dtype": df.dtypes,
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(1),
    "valores_unicos": df.nunique(),
})
resumo

,dtype,nulos,% nulos,valores_unicos
Row ID,int64,0,0.0,51290
Order ID,object,0,0.0,25728
Order Date,object,0,0.0,1430
Ship Date,object,0,0.0,1464
Ship Mode,object,0,0.0,4
Customer ID,object,0,0.0,17415
Customer Name,object,0,0.0,796
Segment,object,0,0.0,3
Postal Code,float64,41296,80.5,631
City,object,0,0.0,3650


In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,51290.0,NaN,NaN,NaN,25645.5,14806.29199,1.0,12823.25,25645.5,38467.75,51290.0
Order ID,51290,25728,CA-2015-SV20365140-42268,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,51290,1430,6/18/2015,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,51290,1464,11/22/2015,130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,51290,4,Standard Class,30775,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,51290,17415,SV-203651406,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,51290,796,Muhammed Yedwab,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,51290,3,Consumer,26518,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Postal Code,9994.0,NaN,NaN,NaN,55190.379428,32063.69335,1040.0,23223.0,56430.5,90008.0,99301.0
City,51290,3650,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Duplicatas

`Row ID` é só um índice sequencial — comparar linhas incluindo essa coluna nunca vai achar duplicata real.
O ideal é comparar pela **chave de negócio**: mesmo pedido (`Order ID`) + mesmo produto (`Product ID`) + mesma quantidade + mesmo valor vendido. Se tudo isso bate, é a mesma linha de pedido duplicada.

In [ ]:
#duplicata "ingênua" (linha inteira, incluindo Row ID) -> nunca aparece, porque Row ID é sempre único
print("Duplicatas de linha inteira:", df.duplicated().sum())

chave_negocio = ["Order ID", "Product ID", "Quantity", "Sales"]
duplicadas = df.duplicated(subset=chave_negocio, keep=False)
print("Linhas envolvidas em duplicatas por chave de negócio:", duplicadas.sum())

df.loc[duplicadas].sort_values(chave_negocio)

Duplicatas de linha inteira: 0
Linhas envolvidas em duplicatas por chave de negócio: 22


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Postal Code,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
30153,44647,AG-2015-CD27903-42236,8/20/2015,8/25/2015,Standard Class,CD-27903,Cynthia Delaney,Home Office,NaN,Oran,...,TEC-AC-5219,Technology,Accessories,"Memorex Mouse, USB",$56.76,2,0.00,$10.74,5.430,High
37874,44649,AG-2015-CD27903-42236,8/20/2015,8/25/2015,Standard Class,CD-27903,Cynthia Delaney,Home Office,NaN,Oran,...,TEC-AC-5219,Technology,Accessories,"Memorex Mouse, USB",$56.76,2,0.00,$10.74,2.770,High
37103,13944,ES-2013-RF1984014-41515,8/29/2013,9/5/2013,Standard Class,RF-1984014,Roy Französisch,Consumer,NaN,Mechelen,...,OFF-AR-5926,Office Supplies,Art,"Sanford Pens, Fluorescent",$37.08,3,0.00,$10.35,2.980,Medium
43214,13946,ES-2013-RF1984014-41515,8/29/2013,9/5/2013,Standard Class,RF-1984014,Roy Französisch,Consumer,NaN,Mechelen,...,OFF-AR-5926,Office Supplies,Art,"Sanford Pens, Fluorescent",$37.08,3,0.00,$10.35,1.840,Medium
34100,10449,ES-2014-SR20740120-41804,6/14/2014,6/21/2014,Standard Class,SR-20740120,Steven Roelle,Home Office,NaN,Madrid,...,OFF-BI-6371,Office Supplies,Binders,"Wilson Jones 3-Hole Punch, Economy",$83.97,3,0.00,$27.63,3.880,Medium
36524,10446,ES-2014-SR20740120-41804,6/14/2014,6/21/2014,Standard Class,SR-20740120,Steven Roelle,Home Office,NaN,Madrid,...,OFF-BI-6371,Office Supplies,Binders,"Wilson Jones 3-Hole Punch, Economy",$83.97,3,0.00,$27.63,3.140,Medium
44705,27443,ID-2012-AR105107-41031,5/2/2012,5/8/2012,Standard Class,AR-105107,Andrew Roberts,Consumer,NaN,Armidale,...,OFF-BI-3736,Office Supplies,Binders,"Cardinal Hole Reinforcements, Recycled",$15.39,3,0.10,$2.52,1.700,Medium
45322,27445,ID-2012-AR105107-41031,5/2/2012,5/8/2012,Standard Class,AR-105107,Andrew Roberts,Consumer,NaN,Armidale,...,OFF-BI-3736,Office Supplies,Binders,"Cardinal Hole Reinforcements, Recycled",$15.39,3,0.10,$2.52,1.640,Medium
4492,29853,IN-2013-RM19375130-41583,11/5/2013,11/6/2013,First Class,RM-19375130,Raymond Messe,Consumer,NaN,Bangkok,...,FUR-CH-5450,Furniture,Chairs,"Office Star Steel Folding Chair, Adjustable",$343.83,5,0.27,$89.43,72.950,Medium
6056,29854,IN-2013-RM19375130-41583,11/5/2013,11/6/2013,First Class,RM-19375130,Raymond Messe,Consumer,NaN,Bangkok,...,FUR-CH-5450,Furniture,Chairs,"Office Star Steel Folding Chair, Adjustable",$343.83,5,0.27,$89.43,55.800,Medium


Encontramos 11 pares duplicados reais (22 linhas) — mesmo pedido, mesmo produto, mesma quantidade e mesmo valor vendido, divergindo só em `Shipping Cost`, `Discount` ou `Profit` por poucos centavos. Por exemplo, o par `Order ID` = AG-2015-CD27903-42236 / `Product ID` = TEC-AC-5219: tudo igual, só o `Shipping Cost` diverge (5.43 vs 2.77) — indica erro de digitação/duplicação no lançamento, não dois itens diferentes.

Importante não confundir com pedidos onde o mesmo cliente comprou o mesmo produto duas vezes em quantidades/valores diferentes (linhas de pedido legítimas) — essas **não** entram na chave de negócio acima e são preservadas.

In [7]:
antes = len(df)
df = df.drop_duplicates(subset=chave_negocio, keep="first").reset_index(drop=True)
print(f"Linhas removidas: {antes - len(df)} | Total agora: {len(df)}")

Linhas removidas: 11 | Total agora: 51279


## 3. Valores faltantes

Só `Postal Code` tem nulos. Antes de decidir o que fazer, é preciso entender **por que** — se for aleatório, dá pra imputar; se for estrutural, imputar seria inventar dado.

In [8]:
df.isna().sum()[df.isna().sum() > 0]

Postal Code    41286
dtype: int64

In [ ]:
#postal Code (CEP norte-americano) só existe para pedidos nos EUA -> ausência é estrutural, não erro de coleta
taxa_nulos_por_pais = df.groupby("Country")["Postal Code"].apply(lambda s: s.isna().mean())
print("EUA:", taxa_nulos_por_pais.get("United States"))
print("Outros países (min/max):", taxa_nulos_por_pais.drop("United States").min(), taxa_nulos_por_pais.drop("United States").max())

EUA: 0.0
Outros países (min/max): 1.0 1.0


Confirmado: 0% de nulos nos EUA e 100% em todos os outros países. `Postal Code` não se aplica fora dos EUA — **não vamos imputar** (colocar 0 ou a moda inventaria um CEP falso). A decisão é manter `NaN` e usar um tipo inteiro que aceite nulo (seção 5).

## 4. Formatos inconsistentes

`Sales` e `Profit` vieram como texto formatado (`$3,709.40`, `$-288.77`) em vez de número. `Product Name` tem espaços extras em algumas linhas.

In [10]:
df[["Sales", "Profit"]].head(3)

,Sales,Profit
0,$221.98,$62.15
1,"$3,709.40",$-288.77
2,"$5,175.17",$919.97


In [ ]:
#remove "$" e separador de milhar "," antes de converter para número
for col in ["Sales", "Profit"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

df[["Sales", "Profit"]].head(3)

,Sales,Profit
0,221.98,62.15
1,3709.40,-288.77
2,5175.17,919.97


In [ ]:
#espaços extras no início/fim em colunas de texto (achado: 16 linhas em Product Name)
colunas_texto = df.select_dtypes(include="object").columns
for col in colunas_texto:
    df[col] = df[col].str.strip()

print("ok: espaços extras removidos de", len(colunas_texto), "colunas de texto")

ok: espaços extras removidos de 17 colunas de texto


## 5. Conversão de tipos

- `Order Date` / `Ship Date`: texto (`11/11/2014`) → `datetime`
- `Sales` / `Profit`: já viraram `float` na seção anterior
- `Postal Code`: `float64` com `NaN` → `Int64` (inteiro anulável do pandas), pra não ficar `73120.0`
- Colunas de baixa cardinalidade (`Ship Mode`, `Segment`, `Category`, `Sub-Category`, `Region`, `Market`, `Order Priority`): `category`, mais leve em memória

In [13]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")

df["Postal Code"] = df["Postal Code"].astype("Int64")

colunas_categoricas = ["Ship Mode", "Segment", "Category", "Sub-Category", "Region", "Market", "Order Priority"]
df[colunas_categoricas] = df[colunas_categoricas].astype("category")

df.dtypes

Row ID                     int64
Order ID                  object
Order Date        datetime64[ns]
Ship Date         datetime64[ns]
Ship Mode               category
Customer ID               object
Customer Name             object
Segment                 category
Postal Code                Int64
City                      object
State                     object
Country                   object
Region                  category
Market                  category
Product ID                object
Category                category
Sub-Category            category
Product Name              object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Shipping Cost            float64
Order Priority          category
dtype: object

## 6. Validação do resultado

Checagens automáticas: se alguma falhar, o `assert` interrompe a célula e aponta o problema.

In [ ]:
#sem duplicatas pela chave de negócio
assert df.duplicated(subset=chave_negocio).sum() == 0, "ainda há duplicatas"

#datas de entrega nunca antes da data do pedido
assert (df["Ship Date"] >= df["Order Date"]).all(), "há entregas antes do pedido"

#valores de venda e quantidade sempre positivos
assert (df["Sales"] > 0).all(), "há Sales <= 0"
assert (df["Quantity"] > 0).all(), "há Quantity <= 0"

#postal Code só é nulo fora dos EUA (nulo estrutural, não erro)
nulos_esperados = (df["Country"] != "United States").sum()
assert df["Postal Code"].isna().sum() == nulos_esperados, "nulos de Postal Code fora do esperado"

print("Todas as validações passaram.")
df.info()

Todas as validações passaram.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51279 entries, 0 to 51278
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          51279 non-null  int64         
 1   Order ID        51279 non-null  object        
 2   Order Date      51279 non-null  datetime64[ns]
 3   Ship Date       51279 non-null  datetime64[ns]
 4   Ship Mode       51279 non-null  category      
 5   Customer ID     51279 non-null  object        
 6   Customer Name   51279 non-null  object        
 7   Segment         51279 non-null  category      
 8   Postal Code     9993 non-null   Int64         
 9   City            51279 non-null  object        
 10  State           51279 non-null  object        
 11  Country         51279 non-null  object        
 12  Region          51279 non-null  category      
 13  Market          51279 non-null  category      
 14  Product ID      51279 no